In [41]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from dotenv import load_dotenv
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
load_dotenv()

True

In [42]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]

 

In [43]:
model = ChatGroq(model="llama-3.3-70b-versatile")

def chat_node(state: ChatState) -> ChatState:
    messages = state["messages"]
    response = model.invoke(messages)
    return {"messages": [response]}



In [44]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

graph.add_node("chat_node", chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)
chatbot = graph.compile(checkpointer=checkpointer)

In [45]:
thread_id = '1'

while True:
    user_input = input("You: ")
    print(f"User: {user_input}")
    if user_input.strip().lower() in ["exit", "quit"]:
        print("Exiting chat.")
        break

    message = [HumanMessage(content=user_input)]

    config = {"configurable": {"thread_id": thread_id}}

    response = chatbot.invoke({"messages": message}, config=config)
    print(f"Chatbot: {response['messages'][-1].content}")

User: Hi my name is hamza
Chatbot: Hello Hamza, it's nice to meet you. Is there something I can help you with or would you like to chat?
User: what is my name
Chatbot: Your name is Hamza.
User: i love to play crickey
Chatbot: That's great, Hamza! Cricket is an amazing sport. Are you a fan of a particular team or player? Do you play cricket yourself, or do you enjoy watching matches?
User: what do I love
Chatbot: You love to play cricket, Hamza!
User: exit
Exiting chat.


In [46]:
chatbot.get_state(config = config)

StateSnapshot(values={'messages': [HumanMessage(content='Hi my name is hamza', additional_kwargs={}, response_metadata={}, id='01ab15e6-b4cb-401a-b9a1-dbc43a5744ba'), AIMessage(content="Hello Hamza, it's nice to meet you. Is there something I can help you with or would you like to chat?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 41, 'total_tokens': 68, 'completion_time': 0.054181071, 'completion_tokens_details': None, 'prompt_time': 0.001940333, 'prompt_tokens_details': None, 'queue_time': 0.015051026, 'total_time': 0.056121404}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019cbd0d-eef4-74e3-a7df-aeee32058973-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 41, 'output_tokens': 27, 'total_tokens': 68}), HumanMessage(content='what is my name', additional_kwargs